In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

In [2]:
datos = pd.read_pickle('../Datos_Limpios/dataset_limpio.pkl')

datos.head()

,ID_Pedido,Fecha_Pedido,Fecha_Envio,Metodo_Envio,ID_Cliente,Nombre_Cliente,Perfil_Cliente,Ciudad,Estado,Pais,Region,Macro_Zona,ID_Producto,Categoria,Subcategoria,Nombre_Producto,Ventas,Cantidad,Descuento,Beneficio,Gastos_Envio,Prioridad_Pedido,Gerente,Devuelto
ID_Venta,,,,,,,,,,,,,,,,,,,,,,,,
40098,CA-2014-AB10015140-41954,2014-11-11,2014-11-13,First Class,AB-100151402,Aaron Bergman,Consumer,Oklahoma City,Oklahoma,United States,Central US,USCA,TEC-PH-5816,Technology,Phones,Samsung Convoy 3,221.98,2,0.0,62.15,40.77,High,Lon Bonher,False
26341,IN-2014-JR162107-41675,2014-02-05,2014-02-07,Second Class,JR-162107,Justin Ritter,Corporate,Wollongong,New South Wales,Australia,Oceania,Asia Pacific,FUR-CH-5379,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",3709.40,9,0.1,-288.77,923.63,Critical,Kauri Anaru,False
25330,IN-2014-CR127307-41929,2014-10-17,2014-10-18,First Class,CR-127307,Craig Reiter,Consumer,Brisbane,Queensland,Australia,Oceania,Asia Pacific,TEC-PH-5356,Technology,Phones,"Nokia Smart Phone, with Caller ID",5175.17,9,0.1,919.97,915.49,Medium,Kauri Anaru,False
13524,ES-2014-KM1637548-41667,2014-01-28,2014-01-30,First Class,KM-1637548,Katherine Murray,Home Office,Berlin,Berlin,Germany,Western Europe,Europe,TEC-PH-5267,Technology,Phones,"Motorola Smart Phone, Cordless",2892.51,5,0.1,-96.54,910.16,Medium,Gilbert Wolff,False
47221,SG-2014-RH9495111-41948,2014-11-05,2014-11-06,Same Day,RH-9495111,Rick Hansen,Consumer,Dakar,Dakar,Senegal,Western Africa,Africa,TEC-CO-6011,Technology,Copiers,"Sharp Wireless Fax, High-Speed",2832.96,8,0.0,311.52,903.04,Critical,Katlego Akosua,False


In [3]:
datos.shape

(51290, 24)

Creamos dos dataframes separando las devoluciones.

In [4]:
sin_devueltos = datos[datos['Devuelto']==False]
sin_devueltos.shape

(49070, 24)

In [5]:
devoluciones = datos[datos['Devuelto']==True]
devoluciones.shape

(2220, 24)

Agrupamos por cantidad de descuento y calculamos el beneficio medio.

Vemos que por encima del 20% empezamos a perder dinero.

In [57]:
sin_devueltos.groupby('Descuento')['Beneficio'].mean()

Descuento
0.000      61.226165
0.002     126.469406
0.070     134.166620
0.100      63.044678
0.150      50.712241
0.170      38.939273
0.200      23.674128
0.202     -14.518537
0.250       5.190979
0.270      -4.463106
0.300     -56.240061
0.320     -88.561481
0.350    -119.892797
0.370     -79.825972
0.400     -45.126092
0.402    -109.882653
0.450     -40.621994
0.470     -43.208735
0.500     -98.444235
0.550    -315.068000
0.570    -551.676364
0.600     -81.216135
0.602    -213.279130
0.650    -309.916000
0.700    -105.530574
0.800    -125.375635
0.850   -1534.330000
Name: Beneficio, dtype: float64

Vemos que paises tienen más y menos beneficio.

In [7]:
beneficio_pais = sin_devueltos.groupby('Pais')['Beneficio'].sum()

In [8]:
beneficio_pais.sort_values(ascending=False).head(10)

Pais
United States     268838.67
China             144278.64
India             125748.23
United Kingdom    109267.83
France            105455.62
Germany           102770.57
Australia         100331.50
Mexico             99028.60
Spain              50110.88
El Salvador        41140.22
Name: Beneficio, dtype: float64

In [55]:
paises_sin_beneficio = beneficio_pais[beneficio_pais<=0]
print (len(paises_sin_beneficio))
paises_sin_beneficio.sort_values().head()

31


Pais
Turkey        -93965.71
Nigeria       -78477.82
Netherlands   -40089.51
Honduras      -28625.29
Pakistan      -21980.66
Name: Beneficio, dtype: float64

Vemos que productos tienen más y menos beneficio.

In [11]:
beneficio_producto = sin_devueltos.groupby('Nombre_Producto')['Beneficio'].sum()

beneficio_producto.sort_values(ascending=False).head(15)

Nombre_Producto
Canon imageCLASS 2200 Advanced Copier                                          25199.94
Motorola Smart Phone, Full Size                                                17104.31
Cisco Smart Phone, Full Size                                                   16978.09
Hoover Stove, Red                                                              11807.96
Sauder Classic Bookcase, Traditional                                           10672.06
Cisco Smart Phone, with Caller ID                                               9786.65
Nokia Smart Phone, with Caller ID                                               9593.26
Harbour Creations Executive Leather Armchair, Adjustable                        9152.53
Nokia Smart Phone, Full Size                                                    8977.56
Belkin Router, USB                                                              8955.01
Hewlett Wireless Fax, High-Speed                                                8492.54
Canon Wireless F

In [13]:
beneficio_producto.sort_values().head()

Nombre_Producto
Cubify CubeX 3D Printer Double Head Print   -9239.97
Lexmark MX611dhe Monochrome Laser Printer   -4589.97
Motorola Smart Phone, Cordless              -4447.04
Cubify CubeX 3D Printer Triple Head Print   -3839.99
Bevis Round Table, Adjustable Height        -3649.90
Name: Beneficio, dtype: float64

Y la cantidad de productos y prodructos sin beneficios.

In [19]:
datos['ID_Producto'].nunique()

3788

In [14]:
producto_sin_beneficio = beneficio_producto [beneficio_producto<=0]
len(producto_sin_beneficio)

694

Creamos el dataframe pedidos.

In [17]:
pedidos = sin_devueltos.groupby('ID_Pedido').agg({
    'Metodo_Envio':'first',
    'Pais':'first',
    'Perfil_Cliente':'first',
    'Ventas':'sum',
    'Beneficio':'sum',
    'Gastos_Envio': 'sum'
})   
pedidos.head()

,Metodo_Envio,Pais,Perfil_Cliente,Ventas,Beneficio,Gastos_Envio
ID_Pedido,,,,,,
AE-2012-PO8865138-41184,Standard Class,United Arab Emirates,Consumer,161.08,-246.08,9.56
AE-2014-EB4110138-41926,Same Day,United Arab Emirates,Consumer,229.00,-236.96,61.18
AE-2015-GH4665138-42351,Standard Class,United Arab Emirates,Consumer,281.51,-429.10,21.38
AE-2015-JD5790138-42070,Standard Class,United Arab Emirates,Consumer,6.43,-11.57,1.49
AE-2015-PG8820138-42313,First Class,United Arab Emirates,Consumer,42.48,-75.06,8.04


Creamos el dataframe pedidos sin beneficio

In [20]:
pedidos_sin_beneficio = pedidos[pedidos['Beneficio']<=0]

In [21]:
pedidos_sin_beneficio.head()

,Metodo_Envio,Pais,Perfil_Cliente,Ventas,Beneficio,Gastos_Envio
ID_Pedido,,,,,,
AE-2012-PO8865138-41184,Standard Class,United Arab Emirates,Consumer,161.08,-246.08,9.56
AE-2014-EB4110138-41926,Same Day,United Arab Emirates,Consumer,229.00,-236.96,61.18
AE-2015-GH4665138-42351,Standard Class,United Arab Emirates,Consumer,281.51,-429.10,21.38
AE-2015-JD5790138-42070,Standard Class,United Arab Emirates,Consumer,6.43,-11.57,1.49
AE-2015-PG8820138-42313,First Class,United Arab Emirates,Consumer,42.48,-75.06,8.04


Calculamos la tasa de pedidos sin beneficio por país.

In [33]:
pedidos_sin_beneficio_por_pais = pedidos_sin_beneficio['Pais'].value_counts()

In [34]:
pedidos_por_pais = pedidos['Pais'].value_counts()

In [54]:
tasa_pedidos_sin_beneficio_por_pais = pedidos_sin_beneficio_por_pais/pedidos_por_pais*100
tasa_pedidos_sin_beneficio_por_pais.sort_values(ascending=False).round(2).head()

Pais
Denmark      100.0
Cyprus       100.0
Laos         100.0
Lithuania    100.0
Ireland      100.0
Name: count, dtype: float64

Creamos una tabla con la tasa de pedidos sin benefio y el total de pedidos por cada país.

In [46]:
sin_beneficio = pd.concat([tasa_pedidos_sin_beneficio_por_pais, pedidos_por_pais], axis=1)
sin_beneficio.head()

,count,count
Pais,,
Afghanistan,NaN,25
Albania,NaN,8
Algeria,NaN,90
Angola,NaN,58
Argentina,90.163934,183


In [50]:
sin_beneficio.columns = ['Tasa', 'Pedidos']

In [60]:
sin_beneficio.sort_values(by='Tasa', ascending=False).head(50)

,Tasa,Pedidos
Pais,,
Denmark,100.000000,31
Cyprus,100.000000,1
Laos,100.000000,1
Lithuania,100.000000,25
Ireland,100.000000,50
United Arab Emirates,100.000000,5
Sweden,100.000000,98
Portugal,100.000000,35
Zimbabwe,100.000000,41


Con estos datos sacamos conclusiones para el EDA.